## ENEM 2021 — Recortes geográficos,  variáveis derivadas (pós‐pré-processamento)

Objetivo: a partir do dataset tratado (nível candidato), criar variáveis derivadas usadas no dashboard/EDA e gerar recortes geográficos (MG e Caxambu), mantendo tipagem consistente e reduzindo categorias não utilizadas.

Entradas: DADOS_TRATADOS_2021 (Parquet, saída do notebook 00-fbps-preprocessamento_2021).
Saídas:

DADOS_TRATADOS_2022_MEDIAS (Parquet)  — arquivo original com as duas variáveis derivadas adicionadas 

DADOS_TRAT_MG_2021 (Parquet) — candidatos com prova em MG + variável regiao

DADOS_TRAT_CAX_2021 (Parquet) — candidatos com prova em Caxambu

(opcional/etapa posterior) tabelas agregadas por município/região (quando você criar/rodar df_agg)

Observação (GitHub): microdados brutos não são versionados; esta etapa opera sobre Parquet já tratado.

In [1]:
import sys
from pathlib import Path

# Permite importar o pacote `src/` a partir do diretório do projeto.
ROOT_PATH = Path().resolve().parents[1]  # notebooks/00_preprocessamento -> projeto
if str(ROOT_PATH) not in sys.path:
    sys.path.append(str(ROOT_PATH))

import re
import numpy as np
import pandas as pd


from src.config import (
    DADOS_TRATADOS_2021, 
    DADOS_TRATADOS_MEDIAS_2021,
    DADOS_TRAT_MG_2021, 
    DADOS_TRAT_CAX_2021
)

from src.preprocessamento.regioes_mg import  atribuir_regiao, MAP_NOME_REGIAO, MAP_REGIOES
from src.preprocessamento.categorias import MAP_SAL_MIN_RENDA_MEDIA


pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")




### 2) Leitura do dataset tratado

In [2]:
df = pd.read_parquet( DADOS_TRATADOS_2021)


df.head(3)

,faixa_etaria,sexo,estado_civil,cor_raca,escola,municipio,uf,nota_cn,nota_ch,nota_lc,nota_mt,lingua,nota_redacao,escolaridade_pai,escolaridade_mae,ocup_pai,ocup_mae,n_pessoas_resd,sal_min,emp_domst,gelad,lv_rp,tv,cel,comptdr,indice_consumo
0,20-25,feminino,solteiro,branca,não informada,Nova Lima,MG,NaN,574.60,472.60,NaN,espanhol,760.00,superior,superior,básico,básico,3,1 a 3,0,1,1,1,1,1,0.18
1,26-35,masculino,solteiro,branca,não informada,Maceió,AL,505.90,551.80,498.30,461.50,espanhol,560.00,até fund,até fund,básico,básico,3,1 a 3,0,1,0,1,2,0,0.10
2,36-45,feminino,divorciado/separado,branca,não informada,Ferraz de Vasconcelos,SP,NaN,NaN,NaN,NaN,espanhol,NaN,até fund,até fund,manual/tec,básico,3,1 a 3,0,1,0,1,2,1,0.24


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3389830 entries, 0 to 3389829
Data columns (total 26 columns):
 #   Column            Dtype   
---  ------            -----   
 0   faixa_etaria      category
 1   sexo              category
 2   estado_civil      category
 3   cor_raca          category
 4   escola            category
 5   municipio         category
 6   uf                category
 7   nota_cn           float32 
 8   nota_ch           float32 
 9   nota_lc           float32 
 10  nota_mt           float32 
 11  lingua            category
 12  nota_redacao      float32 
 13  escolaridade_pai  category
 14  escolaridade_mae  category
 15  ocup_pai          category
 16  ocup_mae          category
 17  n_pessoas_resd    Int8    
 18  sal_min           category
 19  emp_domst         Int8    
 20  gelad             Int8    
 21  lv_rp             Int8    
 22  tv                Int8    
 23  cel               Int8    
 24  comptdr           Int8    
 25  indice_consumo    

### 3) Variáveis derivadas

Nesta etapa são criadas variáveis auxiliares que facilitam análises exploratórias, agregações e etapas posteriores do pipeline, incluindo o dashboard e a modelagem preditiva. Essas variáveis não fazem parte dos microdados originais do ENEM, mas são derivadas de campos já existentes para melhorar a interpretabilidade e a usabilidade dos dados.

#### renda_media (proxy numérica de sal_min)

A variável sal_min representa a renda familiar mensal em faixas de salários mínimos, sendo portanto uma categoria ordinal. Para permitir análises quantitativas simples — como correlações, cálculos de médias, rankings e agregações — foi criada a variável renda_media, que associa a cada faixa um valor numérico representativo, aproximando o ponto médio de cada intervalo.

Essa transformação não pretende estimar a renda real do candidato, mas apenas fornecer uma proxy numérica consistente para análises estatísticas e comparações entre grupos.

#### nota_media (indicador sintético de desempenho)

Para obter uma medida sintética do desempenho do candidato no ENEM, foi criada a variável nota_media, definida como a média aritmética das cinco notas do exame:

* Ciências da Natureza (nota_cn)
* Ciências Humanas (nota_ch)
* Linguagens (nota_lc)
* Matemática (nota_mt)
* Redação (nota_redacao)

Essa variável fornece uma visão geral do desempenho global do candidato, sendo útil para:

* comparações entre grupos socioeconômicos
* análises regionais
* visualizações no dashboard
* definição de variável alvo em experimentos de modelagem preditiva

Embora simplifique o desempenho em um único indicador, a média preserva a escala das notas individuais e facilita análises agregadas mais diretas.

##### Observação metodológica

As notas do ENEM são apresentadas em uma escala comparável entre áreas, o que permite a construção de uma medida sintética de desempenho para fins analíticos. Assim, nota_media não substitui análises específicas por área do conhecimento, mas funciona como um indicador global adequado ao objetivo deste projeto, que é investigar padrões socioeconômicos associados ao desempenho educacional.

In [4]:
#renda média

df["renda_media"] = (
    df["sal_min"].astype("string").map(MAP_SAL_MIN_RENDA_MEDIA).astype("float32")
)

In [5]:
#nota média
df["nota_media"] = df[
    ["nota_cn", "nota_ch", "nota_lc", "nota_mt", "nota_redacao"]
].mean(axis=1).astype("float32")

### 4) Recortes geográficos e limpeza de categorias

In [6]:
# Caxambu
df_caxambu = df.loc[df['municipio'] == 'Caxambu', :].copy()
df_caxambu['municipio'] = df_caxambu['municipio'].cat.remove_unused_categories()
df_caxambu['uf'] = df_caxambu['uf'].cat.remove_unused_categories()
df_caxambu.head(3)

,faixa_etaria,sexo,estado_civil,cor_raca,escola,municipio,uf,nota_cn,nota_ch,nota_lc,nota_mt,lingua,nota_redacao,escolaridade_pai,escolaridade_mae,ocup_pai,ocup_mae,n_pessoas_resd,sal_min,emp_domst,gelad,lv_rp,tv,cel,comptdr,indice_consumo,renda_media,nota_media
3120,até 19,feminino,solteiro,branca,pública,Caxambu,MG,437.60,517.20,541.90,512.10,espanhol,720.00,superior,médio,médio/tec,médio/tec,3,1 a 3,0,1,1,2,3,2,0.25,2.00,545.76
10810,até 19,masculino,solteiro,branca,pública,Caxambu,MG,620.30,602.20,573.20,623.30,inglês,720.00,até fund,até fund,manual/tec,médio/tec,3,1 a 3,0,1,1,2,3,2,0.35,2.00,627.80
13108,até 19,feminino,solteiro,branca,não informada,Caxambu,MG,509.40,564.70,528.20,473.10,inglês,840.00,médio,médio,manual/tec,básico,4,1 a 3,0,1,1,1,3,1,0.25,2.00,583.08


In [7]:
df_mg = df.loc[df['uf'] == 'MG', :].copy()
df_mg['uf'] = df_mg['uf'].cat.remove_unused_categories()
df_mg.head(3)

,faixa_etaria,sexo,estado_civil,cor_raca,escola,municipio,uf,nota_cn,nota_ch,nota_lc,nota_mt,lingua,nota_redacao,escolaridade_pai,escolaridade_mae,ocup_pai,ocup_mae,n_pessoas_resd,sal_min,emp_domst,gelad,lv_rp,tv,cel,comptdr,indice_consumo,renda_media,nota_media
0,20-25,feminino,solteiro,branca,não informada,Nova Lima,MG,NaN,574.60,472.60,NaN,espanhol,760.00,superior,superior,básico,básico,3,1 a 3,0,1,1,1,1,1,0.18,2.00,602.40
8,20-25,feminino,solteiro,negra,pública,Belo Horizonte,MG,487.40,476.50,450.70,493.40,inglês,520.00,até fund,até fund,básico,básico,2,1 a 3,0,1,0,1,1,0,0.15,2.00,485.60
28,20-25,masculino,solteiro,negra,não informada,Contagem,MG,582.30,664.30,576.00,528.70,inglês,580.00,até fund,médio,básico,básico,3,1 a 3,5,1,1,3,3,2,0.32,2.00,586.26


### 5) Variável regiao (macro-regiões dentro de MG)


#### 5.1) Atribuição de região e mapeamento dos nomes para versões mais amigáveis


In [8]:
# Aplicar no dataframe
df_mg["regiao"] = df_mg["municipio"].astype("string").apply(atribuir_regiao)

df_mg["regiao"] = df_mg["regiao"].map(MAP_NOME_REGIAO).astype("category")
df_mg.head(3)




,faixa_etaria,sexo,estado_civil,cor_raca,escola,municipio,uf,nota_cn,nota_ch,nota_lc,nota_mt,lingua,nota_redacao,escolaridade_pai,escolaridade_mae,ocup_pai,ocup_mae,n_pessoas_resd,sal_min,emp_domst,gelad,lv_rp,tv,cel,comptdr,indice_consumo,renda_media,nota_media,regiao
0,20-25,feminino,solteiro,branca,não informada,Nova Lima,MG,NaN,574.60,472.60,NaN,espanhol,760.00,superior,superior,básico,básico,3,1 a 3,0,1,1,1,1,1,0.18,2.00,602.40,Metrop. de Belo Horizonte
8,20-25,feminino,solteiro,negra,pública,Belo Horizonte,MG,487.40,476.50,450.70,493.40,inglês,520.00,até fund,até fund,básico,básico,2,1 a 3,0,1,0,1,1,0,0.15,2.00,485.60,Metrop. de Belo Horizonte
28,20-25,masculino,solteiro,negra,não informada,Contagem,MG,582.30,664.30,576.00,528.70,inglês,580.00,até fund,médio,básico,básico,3,1 a 3,5,1,1,3,3,2,0.32,2.00,586.26,Metrop. de Belo Horizonte


#### 5.2) Diagnóstico: municípios não classificados

In [9]:
todas = set()
for cidades in MAP_REGIOES.values():
    todas.update(cidades)

municipios_df = df_mg["municipio"].astype("string").str.strip().unique()
nao_classificados = [m for m in municipios_df if m not in todas]

print("Municípios únicos em MG:", len(municipios_df))
print("Não classificados:", len(nao_classificados))
# se quiser, mostre só os primeiros 30 para não poluir o notebook
print("Exemplos:", nao_classificados[:30])

Municípios únicos em MG: 188
Não classificados: 0
Exemplos: []


### Exportação
Ao final desta etapa, são salvos três arquivos:

* a base completa com variáveis derivadas (DADOS_TRATADOS_MEDIAS_2021);
* o recorte de Minas Gerais com variável regional (DADOS_TRAT_MG_2021);
* o recorte específico de Caxambu (DADOS_TRAT_CAX_2021).

Esses arquivos servem de entrada para as etapas posteriores de consolidação longitudinal, dashboard e modelagem.

In [10]:
df_mg.to_parquet(DADOS_TRAT_MG_2021, index=False)  
df_caxambu.to_parquet(DADOS_TRAT_CAX_2021, index=False)
df.to_parquet(DADOS_TRATADOS_MEDIAS_2021, index=False)

print("✅ Salvo base completa com médias:", DADOS_TRATADOS_MEDIAS_2021, "| shape:", df.shape)
print("✅ Salvo recorte MG:", DADOS_TRAT_MG_2021, "| shape:", df_mg.shape)
print("✅ Salvo recorte Caxambu:", DADOS_TRAT_CAX_2021, "| shape:", df_caxambu.shape)

✅ Salvo base completa com médias: E:\ciencias_dados\projetos\projeto_enem_ml\dados\intermediarios\dados_tratados_2021_medias.parquet | shape: (3389830, 28)
✅ Salvo recorte MG: E:\ciencias_dados\projetos\projeto_enem_ml\dados\intermediarios\dados_tratados_MG_21.parquet | shape: (327829, 29)
✅ Salvo recorte Caxambu: E:\ciencias_dados\projetos\projeto_enem_ml\dados\intermediarios\dados_tratados_CAX_21.parquet | shape: (830, 28)
